# RF Entropy — Raw Data — Blue Viz

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
sns.set_palette('Blues');plt.rcParams['axes.prop_cycle']=plt.cycler(color=plt.cm.Blues(np.linspace(0.3,0.9,8)))
import warnings;warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('../../data.csv')
if 'Unnamed: 32' in df.columns: df=df.drop(['id','Unnamed: 32'],axis=1)
else: df=df.drop(['id'],axis=1)
df['diagnosis']=df['diagnosis'].map({'M':1,'B':0})
X=df.drop('diagnosis',axis=1);y=df['diagnosis']
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
sc=StandardScaler();X_tr=sc.fit_transform(X_tr);X_te=sc.transform(X_te)

In [ ]:
m=RandomForestClassifier(criterion='entropy',random_state=42).fit(X_tr,y_tr)
yp=m.predict(X_te);ypb=m.predict_proba(X_te)[:,1]
print("RF Entropy trained.")

In [ ]:
acc=accuracy_score(y_te,yp);prec=precision_score(y_te,yp);rec=recall_score(y_te,yp);f1=f1_score(y_te,yp)
cm=confusion_matrix(y_te,yp);tn,fp,fn,tp=cm.ravel();spec=tn/(tn+fp);auc=roc_auc_score(y_te,ypb)
print(f"Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} F1:{f1:.4f} Spec:{spec:.4f} AUC:{auc:.4f}\nCM:\n{cm}")

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['B','M'],yticklabels=['B','M'])
plt.title('RF Entropy — CM (Raw)',fontweight='bold');plt.tight_layout();plt.show()

In [ ]:
n=['Acc','Prec','Rec','F1','Spec'];v=[acc,prec,rec,f1,spec];c=plt.cm.Blues([0.4,0.5,0.6,0.7,0.8])
plt.figure(figsize=(9,5));bars=plt.bar(n,v,color=c,edgecolor='darkblue')
plt.ylim(0,1.05);plt.title('RF Entropy — Metrics (Raw)',fontweight='bold')
for b,v in zip(bars,v):plt.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{v:.4f}',ha='center',fontweight='bold')
plt.tight_layout();plt.show()

In [ ]:
fpr,tpr,_=roc_curve(y_te,ypb)
plt.figure(figsize=(7,6))
plt.plot(fpr,tpr,'royalblue',lw=2.5,label=f'ROC (AUC={auc:.4f})')
plt.fill_between(fpr,tpr,alpha=0.15,color='royalblue')
plt.plot([0,1],[0,1],'gray',ls='--',alpha=0.7)
plt.xlim(0,1);plt.ylim(0,1.05);plt.title('RF Entropy — ROC (Raw)',fontweight='bold')
plt.legend();plt.grid(alpha=0.2);plt.tight_layout();plt.show()